# Teaching a network to copy the heuristic

**Behavior cloning**: record what the hand-written heuristic does in thousands of situations, then train a small network to make the same choices. No rewards, no trial and error yet, just "here's the situation, here's what the expert did, learn to do that." It's plain supervised learning, the same idea as teaching a model to label photos.

Work through it a pocket at a time. Each pocket ends somewhere you can stop:

1. **Record the teacher**: done for you, just run it
2. **Meet PyTorch**: a tiny network, one forward pass
3. **The training loop**: make the loss go down
4. **Is it really learning?**: held-out games, accuracy per action
5. **Let the clone fish**: catches on the real game, against the heuristic's ~14

## Pocket 1: record the teacher

Every time the game asks for a decision, we save two things:

- **the input**: `encode(obs)`, your 40 numbers
- **the answer**: what the heuristic did, as a *type* (0 = wait, 1 = reel, 2 = cast) plus, for casts, *where* (x and y scaled to 0..1)

We record games on seeds **100 to 299**, and keep seeds **1 to 20** unseen for the final test in pocket 5. Testing a model on the same games it learned from would be like grading a test using the practice questions.

The result is saved to `training/data/teacher.npz` (NumPy's file format for arrays), so later pockets can just load it, even after a restart. It takes about half a minute.

In [ ]:
import time
from collections import Counter
from pathlib import Path

import numpy as np

from fishing_env import FishingEnv
from features import encode, action_to_target, ACTION_TYPES, N_FEATURES

TRAIN_SEEDS = range(100, 300)   # 200 games to learn from
DATA = Path("data/teacher.npz")

X, y_type, y_pos, game = [], [], [], []
t0 = time.perf_counter()
with FishingEnv() as env:
    for i, seed in enumerate(TRAIN_SEEDS):
        obs, info = env.reset(seed=seed)
        done = False
        while not done:
            kind, pos = action_to_target(info["teacher"])
            X.append(encode(obs))
            y_type.append(kind)
            y_pos.append(pos)
            game.append(seed)
            obs, reward, done, _, info = env.step(info["teacher"])
        if (i + 1) % 50 == 0:
            print(f"{i + 1} games, {len(X)} decisions, {time.perf_counter() - t0:.0f}s")

DATA.parent.mkdir(exist_ok=True)
np.savez(
    DATA,
    X=np.array(X, dtype=np.float32),
    y_type=np.array(y_type, dtype=np.int64),
    y_pos=np.array(y_pos, dtype=np.float32),
    game=np.array(game, dtype=np.int64),
)
print("saved", DATA)

Now look at what you recorded. Pay attention to how the three action types are split. It'll matter in pocket 4.

In [ ]:
d = np.load(DATA)
X, y_type, y_pos, game = d["X"], d["y_type"], d["y_pos"], d["game"]
print("inputs  X:", X.shape, " (decisions, features)")
print("answers y_type:", y_type.shape, " y_pos:", y_pos.shape)
counts = Counter(y_type.tolist())
for k, name in enumerate(ACTION_TYPES):
    print(f"  {name:5s} {counts[k]:6d}  {100 * counts[k] / len(y_type):5.1f}%")

**Stop here for pocket 1.** You've got a dataset: roughly 40,000 situations, each with the expert's answer. Notice the split: the heuristic waits most of the time. A model that learned only "always wait" would be right most of the time and never catch a fish. Hold that thought until pocket 4.

---

## Pocket 2: meet PyTorch

A **tensor** is PyTorch's array: like a NumPy array, but it can track how every number was computed, and that's what lets it learn. A **network** is a stack of layers. `nn.Linear(40, 64)` takes 40 numbers in and gives 64 out (a grid of 40 × 64 weights, plus 64 biases). `nn.ReLU()` keeps positive numbers and turns negative ones into 0, which is what lets stacked layers learn more than a straight line.

Ours needs **5 outputs**: 3 scores for the action type (wait, reel, cast) and 2 numbers for where to cast.

In [ ]:
import torch
from torch import nn

# TODO: turn the NumPy arrays into tensors
#   torch.from_numpy(X) works, for X, y_type and y_pos

# TODO: build the network: 40 in, two hidden layers of 64 with ReLU, 5 out
#   model = nn.Sequential(
#       nn.Linear(N_FEATURES, 64), nn.ReLU(),
#       ...,
#       nn.Linear(64, 5),
#   )

# TODO: push the first decision through it and print the 5 numbers it gives back.
#   The model wants a batch, so give it X[:1] (one row), not X[0].
#   They'll be meaningless for now. Nothing is trained yet.

---

## Pocket 3: the training loop

Training repeats four lines, over and over:

1. **predict**: run a batch of inputs through the model
2. **measure how wrong it was**: the *loss*. Two parts here:
   - the action type is a choice between classes, so it uses `nn.CrossEntropyLoss`
   - the cast position is two numbers, so it uses `nn.MSELoss` (mean squared error). Only count it on decisions where the teacher actually cast, since position means nothing for wait or reel.
3. **work out how to nudge every weight** to be less wrong: `loss.backward()`
4. **nudge them**: `optimizer.step()`, then `optimizer.zero_grad()` to start fresh

One pass over all the data is an **epoch**. Data goes in **batches** (say 256 rows at a time), shuffled each epoch.

In [ ]:
# TODO: the training loop
#   optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
#   type_loss_fn = nn.CrossEntropyLoss()
#   pos_loss_fn = nn.MSELoss()
#
#   for epoch in range(10):
#       perm = torch.randperm(len(X_t))           # shuffle
#       for start in range(0, len(X_t), 256):
#           idx = perm[start:start + 256]
#           out = model(X_t[idx])                 # shape (batch, 5)
#           type_scores, pos = out[:, :3], out[:, 3:]
#           # loss = type loss + position loss (casts only)
#           # backward, step, zero_grad
#       print(epoch, loss.item())
#
#   Done when: the printed loss goes down epoch after epoch.

---

## Pocket 4: is it really learning?

A falling loss means the model fits the data it trained on, but that's like memorizing practice questions. The real question is whether it does well on **games it has never seen**.

- **Split by game, not by row**: train on most games, hold out a few whole games (the `game` array says which game each row came from). Rows from the same game are near-duplicates, so a random row split would quietly leak answers.
- **Accuracy per action type**: overall accuracy will look great because most answers are "wait." Check how often it gets **reel** and **cast** right on their own. Those are the ones that catch fish.

In [ ]:
# TODO: hold out the last 40 games (seeds 260..299) for validation
#   val = game >= 260 ; train = ~val    (NumPy boolean masks)
#   retrain on train only, then print loss on both
#
# TODO: accuracy per action type on the validation games
#   pred = model(X_val)[:, :3].argmax(dim=1)
#   for each k in 0, 1, 2: of the rows where y_type == k, what fraction did pred get right?

---

## Pocket 5: let the clone fish

Now the real test: the clone plays actual games, on seeds 1 to 20 it never saw. `clone_policy` turns an observation into an action. The evaluation loop below is done for you. It reports average catches, to compare against the heuristic's **about 14** on the same seeds.

In [ ]:
from features import target_to_action


def clone_policy(obs):
    # TODO: encode(obs) -> tensor (one row) -> model -> pick the type with the highest score
    #       -> if it's a cast, use the 2 position numbers -> target_to_action(kind, pos)
    raise NotImplementedError


def evaluate(policy, seeds=range(1, 21)):
    catches = []
    with FishingEnv() as env:
        for seed in seeds:
            obs, info = env.reset(seed=seed)
            done = False
            while not done:
                with torch.no_grad():      # no learning during play
                    action = policy(obs)
                obs, reward, done, _, info = env.step(action)
            catches.append(info["stats"]["catches"])
    return sum(catches) / len(catches), catches


# mean, per_seed = evaluate(clone_policy)
# print(f"clone: {mean:.1f} catches per game (heuristic: about 14)")